# tensorts — Interactive Exploration Notebook

A step-by-step walkthrough of the `tensorts` library: a semantic tensor pipeline for multivariate time series.
Each pipeline step is executed individually so intermediate results can be inspected and plotted.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline

---
## 1. Core: Dim, DimSchema, ChannelIndex, TimeTensor

In [ ]:
from tensorts import Dim, DimSchema, ChannelIndex, TimeTensor

# Dim basics
print("BATCH != TIME:", Dim.BATCH != Dim.TIME)
print("FRAME == WINDOW (alias):", Dim.FRAME == Dim.WINDOW)

# Register a custom dim
custom = Dim.register("MY_SCALE")
print(f"Custom dim: {custom.name}")
print("Idempotent:", Dim.register("MY_SCALE") is custom)

In [ ]:
# DimSchema: axis resolution and duplicate detection
schema = DimSchema((Dim.BATCH, Dim.TIME, Dim.CHANNEL))
print(f"Axis of TIME: {schema.axis(Dim.TIME)}")
print(f"Has CHANNEL: {schema.has(Dim.CHANNEL)}")
print(f"Has FREQUENCY: {schema.has(Dim.FREQUENCY)}")
print(f"Length: {len(schema)}")

# Duplicate detection
try:
    DimSchema((Dim.BATCH, Dim.BATCH))
except ValueError as e:
    print(f"Duplicate detected: {e}")

# Replace a dim
s2 = schema.replace(Dim.TIME, Dim.FREQUENCY)
print(f"After replace TIME->FREQUENCY, axis of FREQUENCY: {s2.axis(Dim.FREQUENCY)}")

In [ ]:
# ChannelIndex: names, groups, globs, slicing
ci = ChannelIndex(
    ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"],
    groups={"acc": "acc_*", "gyro": "gyro_*"},
)
print(f"Channels: {ci.names}")
print(f"acc group indices: {list(ci.group('acc'))}")
print(f"gyro group indices: {list(ci.group('gyro'))}")
print(f"Glob resolve 'acc_*': {ci.resolve_names('acc_*')}")

# Slicing preserves surviving groups
sliced = ci.slice([0, 1, 2])
print(f"\nAfter slicing [0,1,2]: names={sliced.names}, groups={sliced.groups}")

# Auto-naming
auto = ChannelIndex.auto(4, prefix="ch")
print(f"Auto-named: {auto.names}")

In [ ]:
# Dense TimeTensor
data = np.random.randn(8, 100, 4)
y = np.arange(8)
ts = TimeTensor(
    data=data,
    dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["a", "b", "c", "d"],
    y=y,
    metadata={"sampling_rate": 100},
)
print(f"Shape: {ts.shape}")
print(f"Dims: {ts.dims}")
print(f"Channels: {ts.channels.names}")
print(f"y: {ts.y}")
print(f"is_ragged: {ts.is_ragged}")
print(f"n_samples={ts.n_samples}, n_channels={ts.n_channels}")
print(f"metadata: {ts.metadata}")

In [ ]:
# Plot: 2x2 subplot of first sample's 4 channels over time
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.plot(ts.data[0, :, i])
    ax.set_title(f"Channel '{ts.channels.names[i]}'")
    ax.set_ylabel("Value")
axes[1, 0].set_xlabel("Time step")
axes[1, 1].set_xlabel("Time step")
fig.suptitle("Dense TimeTensor — Sample 0, 4 channels", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Ragged TimeTensor
samples_rag = [
    np.random.randn(100, 3),
    np.random.randn(200, 3),
    np.random.randn(150, 3),
]
ts_rag = TimeTensor.from_ragged(samples_rag, channels=["x", "y", "z"], y=np.array([0, 1, 2]))
print(f"is_ragged: {ts_rag.is_ragged}")
print(f"n_samples: {ts_rag.n_samples}")
print(f"lengths: {ts_rag.lengths}")
print(f"shape: {ts_rag.shape}")

# to_dense with padding
dense = ts_rag.to_dense(pad_value=0.0)
print(f"\nAfter to_dense: shape={dense.shape}, is_ragged={dense.is_ragged}")
print(f"Padding check (sample 0, t=100..105, ch0): {dense.data[0, 100:105, 0]}  (should be 0)")

In [ ]:
# Plot: overlay 3 ragged samples for channel 0
fig, ax = plt.subplots(figsize=(10, 4))
for i, s in enumerate(ts_rag._samples):
    ax.plot(s[:, 0], label=f"Sample {i} (len={len(s)})", alpha=0.7)
ax.set_xlabel("Time step")
ax.set_ylabel("Value")
ax.set_title("Ragged TimeTensor — Channel 'x', 3 samples with different lengths")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# with_data preserves metadata, with_y changes labels
ts2 = TimeTensor(
    data=np.random.randn(4, 50, 2),
    dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["a", "b"],
    y=np.array([0, 1, 0, 1]),
    metadata={"sr": 100, "unit": "m/s"},
)
new = ts2.with_data(np.random.randn(4, 50, 2))
print(f"with_data — channels: {new.channels.names}, y: {new.y}, metadata: {new.metadata}")

relabeled = ts2.with_y(np.array([1, 0, 1, 0]))
print(f"with_y — new y: {relabeled.y}, shape unchanged: {relabeled.shape == ts2.shape}")

In [ ]:
# Channel accessor: glob and multi-select
print("ts.ch['a', 'b']:", ts.ch["a", "b"].shape, "channels:", ts.ch["a", "b"].channels.names)
print("ts.ch['a*']:    ", ts.ch["a*"].shape, "channels:", ts.ch["a*"].channels.names)

# Batch slicing
print(f"\nts[0] shape: {ts[0].shape}, y: {ts[0].y}")
print(f"ts[2:5] shape: {ts[2:5].shape}, y: {ts[2:5].y}")
print(f"ts[[0,3,7]] shape: {ts[[0,3,7]].shape}")

---
## 2. Y / Label Handling

In [ ]:
from tensorts import Pipeline, StandardScaler, Flatten, temporal_split

# Pattern 1 (native): y embedded in TimeTensor
data_p1 = np.random.randn(10, 50, 3)
y_p1 = np.array([0, 1, 2, 0, 1, 2, 0, 1, 2, 0])
X_p1 = TimeTensor(data=data_p1, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL), y=y_p1)

pipe = Pipeline([("scale", StandardScaler()), ("flat", Flatten())])
Xt = pipe.fit_transform(X_p1)  # no y argument needed
print(f"Input y:  {X_p1.y}")
print(f"Output y: {Xt.y}")
print(f"y survived pipeline: {np.array_equal(Xt.y, y_p1)}")

# Train/test split preserves y
train, test = temporal_split(X_p1, test_ratio=0.2)
print(f"\nTrain y: {train.y}")
print(f"Test y:  {test.y}")

In [ ]:
from tensorts import SklearnWrapper
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Pattern 2 (sklearn): X + y passed separately for cross_val_score
data_p2 = np.random.randn(30, 20, 3)
y_p2 = np.random.randint(0, 3, 30)
X_p2 = TimeTensor(data=data_p2, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL), y=y_p2)

pipe2 = Pipeline([("scale", StandardScaler()), ("flat", Flatten())])
wrapper = SklearnWrapper(pipe2, LogisticRegression(max_iter=200))

scores = cross_val_score(wrapper, X_p2, y=X_p2.y, cv=3)
print(f"CV scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")

In [ ]:
from tensorts import SlidingWindow

# SlidingWindow label propagation
samples_sw = [np.random.randn(100, 2), np.random.randn(150, 2)]
y_sw = np.array([0, 1])
X_sw = TimeTensor.from_ragged(samples_sw, y=y_sw)

win = SlidingWindow(size=50, stride=25, label_strategy="repeat")
Xt_sw = win.fit_transform(X_sw)

counts = Xt_sw.metadata.get("_window_counts")
print(f"Window counts per sample: {counts}")
print(f"Total windows: {Xt_sw.n_samples}")
print(f"Expanded y: {Xt_sw.y}")
print(f"All windows from sample 0 have label 0: {all(Xt_sw.y[:counts[0]] == 0)}")
print(f"All windows from sample 1 have label 1: {all(Xt_sw.y[counts[0]:] == 1)}")

In [ ]:
from tensorts import LabelEncoder

# LabelEncoder: string labels -> integers
data_le = np.random.randn(9, 10, 2)
y_le = np.array(["cat", "dog", "cat", "bird", "dog", "bird", "cat", "dog", "bird"])
X_le = TimeTensor(data=data_le, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL), y=y_le)

enc = LabelEncoder()
Xt_le = enc.fit_transform(X_le)
print(f"Original y: {y_le}")
print(f"Encoded y:  {Xt_le.y}")
print(f"Classes:    {enc.classes_}")

---
## 3. IO Utilities

In [ ]:
from tensorts import from_numpy, to_numpy, to_sklearn, to_dataframe

# from_numpy
data_io = np.random.randn(5, 30, 3)
ts_io = from_numpy(data_io, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                   channels=["a", "b", "c"], y=np.arange(5))
print(f"from_numpy: shape={ts_io.shape}, channels={ts_io.channels.names}")

# to_numpy
arr = to_numpy(ts_io)
print(f"to_numpy: type={type(arr).__name__}, shape={arr.shape}")

# to_sklearn
flat_io = Flatten().fit_transform(ts_io)
X_np, y_np = to_sklearn(flat_io)
print(f"to_sklearn: X={X_np.shape}, y={y_np.shape}")

In [ ]:
# to_dataframe
data_df = np.arange(12).reshape(1, 4, 3).astype(float)
ts_df = TimeTensor(data=data_df, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                   channels=["x", "y", "z"])
df = to_dataframe(ts_df, sample_index=0)
print(f"DataFrame shape: {df.shape}, columns: {list(df.columns)}")
df

---
## 4. Preprocessing Transforms (step-by-step)

In [ ]:
from tensorts import StandardScaler, MinMaxScaler, RobustScaler, Interpolate

# StandardScaler on dense data
data_ss = np.random.randn(32, 100, 4) * 10 + 5
ts_ss = TimeTensor(data=data_ss, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                   channels=["a", "b", "c", "d"])
result_ss = StandardScaler().fit_transform(ts_ss)

# Plot before/after histograms
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for c in range(4):
    axes[0, c].hist(ts_ss.data[:, :, c].ravel(), bins=40, alpha=0.7, color='steelblue')
    axes[0, c].set_title(f"Before — ch '{ts_ss.channels.names[c]}'")
    axes[1, c].hist(result_ss.data[:, :, c].ravel(), bins=40, alpha=0.7, color='coral')
    axes[1, c].set_title(f"After — ch '{ts_ss.channels.names[c]}'")
    print(f"Channel {c}: mean={result_ss.data[:,:,c].mean():.4f}, std={result_ss.data[:,:,c].std():.4f}")
fig.suptitle("StandardScaler — Before vs After", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# StandardScaler on ragged data
samples_ssr = [np.random.randn(np.random.randint(50, 150), 2) * 5 + 3 for _ in range(20)]
ts_ssr = TimeTensor.from_ragged(samples_ssr, channels=["a", "b"])
result_ssr = StandardScaler().fit_transform(ts_ssr)
print(f"Still ragged: {result_ssr.is_ragged}")
all_data_ssr = np.concatenate(result_ssr._samples, axis=0)
print(f"Pooled mean ch0: {all_data_ssr[:, 0].mean():.4f}")
print(f"Pooled std ch0:  {all_data_ssr[:, 0].std():.4f}")

In [ ]:
# MinMaxScaler
data_mm = np.random.randn(16, 50, 3) * 100
ts_mm = TimeTensor(data=data_mm, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))
print(f"Before: min={ts_mm.data.min():.2f}, max={ts_mm.data.max():.2f}")
result_mm = MinMaxScaler(feature_range=(0, 1)).fit_transform(ts_mm)
print(f"After:  min={result_mm.data.min():.4f}, max={result_mm.data.max():.4f}")

In [ ]:
# RobustScaler with outlier
data_rs = np.random.randn(16, 50, 2) * 10
data_rs[0, 0, 0] = 1000  # inject outlier
ts_rs = TimeTensor(data=data_rs, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))
result_rs = RobustScaler().fit_transform(ts_rs)
print(f"Outlier value before: {data_rs[0, 0, 0]:.0f}")
print(f"Median of ch0 after RobustScaler: {np.median(result_rs.data[:, :, 0]):.4f} (should be ~0)")

In [ ]:
# Interpolate: NaN filling
data_interp = np.array([[[1, 2], [np.nan, 4], [5, np.nan], [7, 8]]], dtype=float)
ts_interp = TimeTensor(data=data_interp, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                       channels=["a", "b"])
result_interp = Interpolate(method="linear").fit_transform(ts_interp)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for c in range(2):
    axes[c].plot(ts_interp.data[0, :, c], 'o--', label='Before (with NaN)', markersize=8)
    axes[c].plot(result_interp.data[0, :, c], 's-', label='After (interpolated)', markersize=8)
    axes[c].set_title(f"Channel '{ts_interp.channels.names[c]}'")
    axes[c].legend()
    axes[c].set_xlabel("Time step")
fig.suptitle("Interpolate — Linear NaN Filling", fontsize=13)
plt.tight_layout()
plt.show()
print(f"NaNs remaining: {np.isnan(result_interp.data).sum()}")

---
## 5. Temporal Transforms (step-by-step)

In [ ]:
from tensorts import SlidingWindow, Pad, Truncate, Resample

# SlidingWindow on dense data
data_sw = np.arange(20, dtype=float).reshape(1, 20, 1)
ts_sw = TimeTensor(data=data_sw, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))
result_sw = SlidingWindow(size=5, stride=5).fit_transform(ts_sw)
print(f"Input shape:  {ts_sw.shape}")
print(f"Output shape: {result_sw.shape}  (B, K_windows, W, C)")
print(f"Has WINDOW dim: {result_sw.has_dim(Dim.WINDOW)}")
print(f"Window 0: {result_sw.data[0, 0, :, 0]}")
print(f"Window 1: {result_sw.data[0, 1, :, 0]}")

In [ ]:
# SlidingWindow on ragged data with label propagation
samples_swr = [np.random.randn(100, 3), np.random.randn(200, 3)]
y_swr = np.array([0, 1])
ts_swr = TimeTensor.from_ragged(samples_swr, channels=["a", "b", "c"], y=y_swr)
result_swr = SlidingWindow(size=50, stride=25, label_strategy="repeat").fit_transform(ts_swr)

counts_swr = result_swr.metadata.get("_window_counts")
print(f"Input: {ts_swr.n_samples} ragged samples, lengths={ts_swr.lengths}")
print(f"Output: {result_swr.shape} (dense)")
print(f"Window counts per original sample: {counts_swr}")
print(f"\nLabel propagation table:")
print(f"  Windows 0..{counts_swr[0]-1} -> label {result_swr.y[0]}")
print(f"  Windows {counts_swr[0]}..{result_swr.n_samples-1} -> label {result_swr.y[counts_swr[0]]}")

In [ ]:
# Pad ragged to dense
samples_pad = [np.ones((80, 2)), np.ones((120, 2))]
ts_pad = TimeTensor.from_ragged(samples_pad, channels=["a", "b"])
result_pad = Pad(target_length=150, mode="constant", value=0.0).fit_transform(ts_pad)
print(f"Before: ragged={ts_pad.is_ragged}, lengths={ts_pad.lengths}")
print(f"After:  shape={result_pad.shape}, ragged={result_pad.is_ragged}")

# Truncate
data_tr = np.random.randn(4, 100, 3)
ts_tr = TimeTensor(data=data_tr, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))
result_tr = Truncate(target_length=60).fit_transform(ts_tr)
print(f"\nTruncate: {ts_tr.shape} -> {result_tr.shape}")

In [ ]:
# Resample ragged to uniform length
samples_res = [np.random.randn(100, 2), np.random.randn(200, 2), np.random.randn(150, 2)]
ts_res = TimeTensor.from_ragged(samples_res, channels=["x", "y"])

print(f"Before: ragged, lengths={ts_res.lengths}")
result_res = Resample(target_length=64).fit_transform(ts_res)
print(f"After:  shape={result_res.shape}, ragged={result_res.is_ragged}")

# Plot before/after for sample 0
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ts_res._samples[0][:, 0])
axes[0].set_title(f"Before resample (len={len(ts_res._samples[0])})")
axes[0].set_xlabel("Time step")
axes[1].plot(result_res.data[0, :, 0])
axes[1].set_title(f"After resample (len={result_res.shape[1]})")
axes[1].set_xlabel("Time step")
fig.suptitle("Resample — Sample 0, Channel 'x'", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Shape Transforms

In [ ]:
from tensorts import Flatten, Transpose, Squeeze, Unsqueeze, PCA

# Flatten
data_fl = np.random.randn(4, 10, 5, 3)
ts_fl = TimeTensor(data=data_fl, dims=(Dim.BATCH, Dim.WINDOW, Dim.TIME, Dim.CHANNEL),
                   channels=["a", "b", "c"])
result_fl = Flatten().fit_transform(ts_fl)
print(f"Flatten: {ts_fl.shape} -> {result_fl.shape}  (10*5*3 = 150)")

# Transpose
data_tp = np.random.randn(4, 8, 3)
ts_tp = TimeTensor(data=data_tp, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))
result_tp = Transpose(order=(Dim.BATCH, Dim.CHANNEL, Dim.TIME)).fit_transform(ts_tp)
print(f"Transpose: {ts_tp.shape} -> {result_tp.shape}, dims={result_tp.dims}")

# Squeeze / Unsqueeze
data_sq = np.random.randn(4, 1, 10, 3)
ts_sq = TimeTensor(data=data_sq, dims=(Dim.BATCH, Dim.WINDOW, Dim.TIME, Dim.CHANNEL))
squeezed = Squeeze(dim=Dim.WINDOW).fit_transform(ts_sq)
print(f"Squeeze:   {ts_sq.shape} -> {squeezed.shape}")
unsqueezed = Unsqueeze(dim=Dim.WINDOW, position=1).fit_transform(squeezed)
print(f"Unsqueeze: {squeezed.shape} -> {unsqueezed.shape}")

In [ ]:
# PCA: reduce to 2 components and plot explained variance
rng = np.random.RandomState(0)
latent = rng.randn(60, 2)
mixing = rng.randn(2, 8)
data_pca = (latent @ mixing + rng.randn(60, 8) * 0.01)[:, np.newaxis, :]
ts_pca = TimeTensor(data=data_pca, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))

pca = PCA(n_components=2)
result_pca = pca.fit_transform(ts_pca)
print(f"PCA: {ts_pca.shape} -> {result_pca.shape}")
print(f"Component channels: {result_pca.channels.names}")

# Plot explained variance
fig, ax = plt.subplots(figsize=(8, 4))
evr = pca._pca.explained_variance_ratio_
ax.bar(range(len(evr)), evr, color='steelblue', alpha=0.8)
ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance Ratio")
ax.set_title(f"PCA Explained Variance (total: {sum(evr):.4f})")
ax.set_xticks(range(len(evr)))
plt.tight_layout()
plt.show()

---
## 7. Spectral Transforms (step-by-step with plots)

In [ ]:
from tensorts import PreEmphasis, ButterworthFilter, BandpassFilter, STFT, MelSpectrogram, LogTransform

# PreEmphasis
t_pe = np.linspace(0, 4 * np.pi, 200)
data_pe = np.sin(t_pe).reshape(1, 200, 1)
ts_pe = TimeTensor(data=data_pe, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))
result_pe = PreEmphasis(coeff=0.97).fit_transform(ts_pe)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ts_pe.data[0, :, 0], label='Original', alpha=0.8)
ax.plot(result_pe.data[0, :, 0], label='Pre-emphasized (coeff=0.97)', alpha=0.8)
ax.set_xlabel("Time step")
ax.set_ylabel("Value")
ax.set_title("PreEmphasis — Original vs Emphasized")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ButterworthFilter: 10Hz + 200Hz, lowpass at 50Hz
sr = 1000
t_bf = np.linspace(0, 1, sr)
signal_10 = np.sin(2 * np.pi * 10 * t_bf)
signal_200 = 0.5 * np.sin(2 * np.pi * 200 * t_bf)
signal_mix = signal_10 + signal_200
data_bf = signal_mix[np.newaxis, :, np.newaxis]
ts_bf = TimeTensor(data=data_bf, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                   metadata={"sampling_rate": sr})

lpf = ButterworthFilter(cutoff=50, btype="lowpass", sampling_rate=sr)
result_bf = lpf.fit_transform(ts_bf)

# FFT analysis
fft_in = np.abs(np.fft.rfft(data_bf[0, :, 0]))
fft_out = np.abs(np.fft.rfft(result_bf.data[0, :, 0]))
freqs = np.fft.rfftfreq(sr, d=1/sr)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(t_bf[:200], data_bf[0, :200, 0])
axes[0, 0].set_title("Before — Time Domain")
axes[0, 0].set_xlabel("Time (s)")
axes[0, 1].plot(freqs[:250], fft_in[:250])
axes[0, 1].set_title("Before — FFT Magnitude")
axes[0, 1].set_xlabel("Frequency (Hz)")
axes[0, 1].axvline(50, color='r', linestyle='--', label='Cutoff 50Hz')
axes[0, 1].legend()
axes[1, 0].plot(t_bf[:200], result_bf.data[0, :200, 0])
axes[1, 0].set_title("After Lowpass — Time Domain")
axes[1, 0].set_xlabel("Time (s)")
axes[1, 1].plot(freqs[:250], fft_out[:250])
axes[1, 1].set_title("After Lowpass — FFT Magnitude")
axes[1, 1].set_xlabel("Frequency (Hz)")
axes[1, 1].axvline(50, color='r', linestyle='--', label='Cutoff 50Hz')
axes[1, 1].legend()
fig.suptitle("ButterworthFilter — 10Hz + 200Hz signal, lowpass at 50Hz", fontsize=13)
plt.tight_layout()
plt.show()

idx_200 = np.argmin(np.abs(freqs - 200))
print(f"200Hz attenuation: {fft_out[idx_200]/fft_in[idx_200]:.4f} (should be << 1)")

In [ ]:
# BandpassFilter
sr_bp = 500
data_bp = np.random.randn(2, sr_bp, 3)
ts_bp = TimeTensor(data=data_bp, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                   metadata={"sampling_rate": sr_bp})
result_bp = BandpassFilter(low=1, high=20, sampling_rate=sr_bp).fit_transform(ts_bp)
print(f"BandpassFilter: {ts_bp.shape} -> {result_bp.shape} (shape preserved)")

In [ ]:
# STFT
sr_stft = 8000
data_stft = np.random.randn(2, sr_stft, 2)
ts_stft = TimeTensor(data=data_stft, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))
result_stft = STFT(n_fft=256, hop_length=128).fit_transform(ts_stft)
print(f"STFT: {ts_stft.shape} -> {result_stft.shape}")
print(f"Has FREQUENCY dim: {result_stft.has_dim(Dim.FREQUENCY)}")
print(f"Freq bins: {result_stft.shape[1]} (n_fft/2+1 = {256//2+1})")

# Plot spectrogram for sample 0, channel 0
fig, ax = plt.subplots(figsize=(10, 4))
spec = result_stft.data[0, :, :, 0]  # (freq, time)
im = ax.imshow(spec, aspect='auto', origin='lower', cmap='viridis')
ax.set_xlabel("Time Frame")
ax.set_ylabel("Frequency Bin")
ax.set_title("STFT Spectrogram — Sample 0, Channel 0")
plt.colorbar(im, ax=ax, label='Magnitude')
plt.tight_layout()
plt.show()

In [ ]:
# MelSpectrogram
sr_mel = 16000
data_mel = np.random.randn(4, sr_mel, 1)
ts_mel = TimeTensor(data=data_mel, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                    metadata={"sampling_rate": sr_mel})
result_mel = MelSpectrogram(n_fft=512, hop_length=160, n_mels=64,
                            sampling_rate=sr_mel).fit_transform(ts_mel)
print(f"MelSpectrogram: {ts_mel.shape} -> {result_mel.shape}")
print(f"Mel bins: {result_mel.shape[1]}, Has FREQUENCY dim: {result_mel.has_dim(Dim.FREQUENCY)}")

# Plot mel spectrogram
fig, ax = plt.subplots(figsize=(10, 4))
mel_data = result_mel.data[0, :, :, 0]  # (mel_bins, time)
im = ax.imshow(mel_data, aspect='auto', origin='lower', cmap='magma')
ax.set_xlabel("Time Frame")
ax.set_ylabel("Mel Bin")
ax.set_title("Mel Spectrogram — Sample 0, Channel 0")
plt.colorbar(im, ax=ax, label='Magnitude')
plt.tight_layout()
plt.show()

In [ ]:
# LogTransform
data_log = np.abs(np.random.randn(4, 50, 3)) + 0.01
ts_log = TimeTensor(data=data_log, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))
result_log = LogTransform(eps=1e-10).fit_transform(ts_log)
print(f"LogTransform: {ts_log.shape} -> {result_log.shape}")
print(f"Before range: [{ts_log.data.min():.4f}, {ts_log.data.max():.4f}]")
print(f"After range:  [{result_log.data.min():.4f}, {result_log.data.max():.4f}]")

---
## 8. Pipeline + ChannelTransform + ChannelSelect

In [ ]:
# Pipeline fit_transform with y preserved
data_pipe = np.random.randn(16, 50, 4)
ts_pipe = TimeTensor(data=data_pipe, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                     y=np.random.randint(0, 3, 16))
pipe = Pipeline([("scale", StandardScaler()), ("flat", Flatten())])
result_pipe = pipe.fit_transform(ts_pipe)
print(f"Pipeline: {ts_pipe.shape} -> {result_pipe.shape}")
print(f"y preserved: {result_pipe.y is not None}, len(y)={len(result_pipe.y)}")

In [ ]:
# Pipeline fit on train, transform test (no leakage)
data_pipe2 = np.random.randn(100, 50, 4) * 10
ts_pipe2 = TimeTensor(data=data_pipe2, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                      y=np.random.randint(0, 3, 100))
train_p, test_p = temporal_split(ts_pipe2, test_ratio=0.2)

pipe2 = Pipeline([("scale", StandardScaler()), ("flat", Flatten())])
Xt_train = pipe2.fit_transform(train_p)
Xt_test = pipe2.transform(test_p)  # uses train statistics
print(f"Train: {Xt_train.shape}, Test: {Xt_test.shape}")
print(f"Same feature count: {Xt_train.shape[1] == Xt_test.shape[1]}")

In [ ]:
# Pipeline get_params / set_params
pipe3 = Pipeline([("scale", StandardScaler()), ("flat", Flatten())])
params = pipe3.get_params(deep=True)
print("Params:", {k: v for k, v in params.items() if 'scale' in k})
pipe3.set_params(scale__eps=1e-4)
print(f"After set_params: scale.eps = {pipe3['scale'].eps}")

In [ ]:
from tensorts import ChannelTransform, ChannelSelect

# ChannelTransform: different scalers per group
data_ct = np.random.randn(8, 100, 4) * np.array([1, 10, 100, 1000])
ts_ct = TimeTensor(
    data=data_ct, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["a", "b", "c", "d"],
    channel_groups={"ab": ["a", "b"], "cd": ["c", "d"]},
)
ct = ChannelTransform({"ab": StandardScaler(), "cd": MinMaxScaler()})
result_ct = ct.fit_transform(ts_ct)

ab = result_ct.ch["a", "b"]
cd = result_ct.ch["c", "d"]
print(f"Group 'ab' (StandardScaler): mean={np.mean(ab.data):.4f}, std={np.std(ab.data):.4f}")
print(f"Group 'cd' (MinMaxScaler):   min={cd.data.min():.4f}, max={cd.data.max():.4f}")

In [ ]:
# ChannelTransform remainder='drop'
data_ctd = np.random.randn(4, 50, 4)
ts_ctd = TimeTensor(
    data=data_ctd, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["a", "b", "c", "d"],
    channel_groups={"ab": ["a", "b"]},
)
ct_drop = ChannelTransform({"ab": StandardScaler()}, remainder="drop")
result_ctd = ct_drop.fit_transform(ts_ctd)
print(f"remainder='drop': {ts_ctd.n_channels} channels -> {result_ctd.n_channels} channels")
print(f"Surviving channels: {result_ctd.channels.names}")

In [ ]:
# ChannelSelect: transform subset, passthrough others
data_cs = np.random.randn(8, 100, 6) * 10
ts_cs = TimeTensor(
    data=data_cs, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["a_x", "a_y", "a_z", "g_x", "g_y", "g_z"],
)
cs = ChannelSelect("a_*", StandardScaler())
result_cs = cs.fit_transform(ts_cs)

a_mean = np.mean(result_cs.ch["a_*"].data)
g_unchanged = np.array_equal(result_cs.ch["g_*"].data, ts_cs.ch["g_*"].data)
print(f"All 6 channels present: {result_cs.n_channels == 6}")
print(f"a_* channels scaled (mean~0): {a_mean:.4f}")
print(f"g_* channels unchanged: {g_unchanged}")

---
## 9. End-to-End Pipelines (Step by Step)

Each pipeline step is executed individually to inspect intermediate results.

### Pipeline A: Air Quality (dense)
Interpolate -> StandardScaler -> SlidingWindow -> Statistical Features

In [ ]:
from tensorts import Interpolate, StandardScaler, SlidingWindow, Flatten, make_statistical_extractor

# Create air quality data with NaNs
B, T, C = 2, 500, 3
data_aq = np.random.randn(B, T, C) * 10
mask = np.random.random((B, T, C)) < 0.02
data_aq[mask] = np.nan
X_aq = TimeTensor(data=data_aq, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                  channels=["PM2.5", "PM10", "O3"])
print(f"Input: shape={X_aq.shape}, NaN count={np.isnan(X_aq.data).sum()}")

In [ ]:
# Step 1: Interpolate
X1 = Interpolate(method="linear").fit_transform(X_aq)
print(f"After Interpolate: shape={X1.shape}, NaNs={np.isnan(X1.data).sum()}")

In [ ]:
# Step 2: StandardScaler
scaler_aq = StandardScaler()
X2 = scaler_aq.fit_transform(X1)
print(f"After StandardScaler: shape={X2.shape}")
for c in range(C):
    print(f"  Channel '{X2.channels.names[c]}': mean={X2.data[:,:,c].mean():.4f}, std={X2.data[:,:,c].std():.4f}")

In [ ]:
# Step 3: SlidingWindow
X3 = SlidingWindow(size=50, stride=25, label_strategy="repeat").fit_transform(X2)
print(f"After SlidingWindow: shape={X3.shape}, dims={X3.dims}")

In [ ]:
# Step 4: Statistical features
X4 = make_statistical_extractor(["mean", "std", "min", "max"]).fit_transform(X3)
print(f"After stats: shape={X4.shape}")

# Plot heatmap of features for first sample
fig, ax = plt.subplots(figsize=(10, 4))
# Reshape for visualization: (windows, stats*channels)
feat_2d = X4.data[0].reshape(X4.shape[1], -1)
im = ax.imshow(feat_2d.T, aspect='auto', cmap='coolwarm')
ax.set_xlabel("Window index")
ax.set_ylabel("Feature")
ax.set_title("Air Quality Pipeline — Feature Heatmap (Sample 0)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Pipeline B: HAR (ragged)
Per-group ChannelTransform -> SlidingWindow -> Statistical Features -> Flatten

In [ ]:
from tensorts import ChannelTransform, RobustScaler, FeatureExtractor

# Create ragged HAR data
rng_har = np.random.default_rng(0)
samples_har = [rng_har.standard_normal((rng_har.integers(150, 400), 6)) for _ in range(30)]
labels_har = np.tile([0, 1, 2], 10)
ts_har = TimeTensor.from_ragged(
    samples_har,
    channels=["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"],
    channel_groups={"acc": "acc_*", "gyro": "gyro_*"},
    y=labels_har,
)
print(f"Input: {ts_har.n_samples} ragged samples")
print(f"Lengths (first 5): {ts_har.lengths[:5]}")
print(f"Labels: {labels_har}")

In [ ]:
# Step 1: Per-group scaling (ragged in, ragged out)
ct_har = ChannelTransform({"acc": StandardScaler(), "gyro": RobustScaler()})
H1 = ct_har.fit_transform(ts_har)
print(f"After ChannelTransform: ragged={H1.is_ragged}, n_samples={H1.n_samples}")
print(f"y preserved: {np.array_equal(H1.y, labels_har)}")

In [ ]:
# Step 2: SlidingWindow (ragged -> dense, this is where ragged->dense transition happens)
H2 = SlidingWindow(size=64, stride=32, label_strategy="repeat").fit_transform(H1)
print(f"After SlidingWindow: shape={H2.shape} (now DENSE)")
print(f"Total windows: {H2.n_samples}")
print(f"y expanded: len={len(H2.y)}, unique={np.unique(H2.y)}")

In [ ]:
# Step 3: Statistical features
H3 = make_statistical_extractor(["mean", "std", "rms", "zcr"]).fit_transform(H2)
print(f"After stats: shape={H3.shape}")

# Step 4: Flatten
H4 = Flatten().fit_transform(H3)
print(f"After Flatten: shape={H4.shape} (4 stats x 6 channels = {4*6} features)")
print(f"y still present: {H4.y is not None}, len={len(H4.y)}")

### Pipeline C: Audio Mel (dense)
PreEmphasis -> MelSpectrogram -> LogTransform -> StandardScaler

In [ ]:
# Create audio data
sr_audio = 16000
data_audio = np.random.randn(4, sr_audio, 1)
ts_audio = TimeTensor(data=data_audio, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                      channels=["audio"], y=np.array([0, 1, 0, 1]),
                      metadata={"sampling_rate": sr_audio})
print(f"Input: shape={ts_audio.shape}, sr={sr_audio}Hz")

In [ ]:
# Step 1: PreEmphasis
A1 = PreEmphasis(coeff=0.97).fit_transform(ts_audio)
print(f"After PreEmphasis: shape={A1.shape}")

In [ ]:
# Step 2: MelSpectrogram
A2 = MelSpectrogram(n_fft=512, hop_length=160, n_mels=64,
                    sampling_rate=sr_audio).fit_transform(A1)
print(f"After MelSpectrogram: shape={A2.shape}")
print(f"Mel bins: {A2.shape[1]}, Time frames: {A2.shape[2]}")

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(A2.data[0, :, :, 0], aspect='auto', origin='lower', cmap='magma')
ax.set_title("After MelSpectrogram — Sample 0")
ax.set_xlabel("Time Frame")
ax.set_ylabel("Mel Bin")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Step 3: LogTransform
A3 = LogTransform().fit_transform(A2)
print(f"After LogTransform: shape={A3.shape}")
print(f"Value range: [{A3.data.min():.2f}, {A3.data.max():.2f}]")

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(A3.data[0, :, :, 0], aspect='auto', origin='lower', cmap='magma')
ax.set_title("After LogTransform — Sample 0")
ax.set_xlabel("Time Frame")
ax.set_ylabel("Mel Bin")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: StandardScaler
A4 = StandardScaler().fit_transform(A3)
print(f"After StandardScaler: shape={A4.shape}")
print(f"y preserved: {np.array_equal(A4.y, [0, 1, 0, 1])}")

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(A4.data[0, :, :, 0], aspect='auto', origin='lower', cmap='magma')
ax.set_title("After StandardScaler — Sample 0 (final)")
ax.set_xlabel("Time Frame")
ax.set_ylabel("Mel Bin")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Pipeline D: Multi-fn FeatureExtractor in windowed pipeline
StandardScaler -> SlidingWindow -> FeatureExtractor (with custom autocorr) -> Flatten

In [ ]:
from tensorts import FeatureExtractor

# Custom autocorrelation function
def autocorr_lag1(x):
    if len(x) < 2:
        return np.float64(0.0)
    return np.float64(np.corrcoef(x[:-1], x[1:])[0, 1])

# Create ragged data
samples_fe = [np.random.randn(200 + i * 30, 4) for i in range(20)]
y_fe = np.random.randint(0, 3, 20)
ts_fe = TimeTensor.from_ragged(samples_fe, y=y_fe, channels=["a", "b", "c", "d"])
print(f"Input: {ts_fe.n_samples} ragged samples, lengths range: [{min(ts_fe.lengths)}, {max(ts_fe.lengths)}]")

In [ ]:
# Step 1: StandardScaler (ragged)
D1 = StandardScaler().fit_transform(ts_fe)
print(f"After StandardScaler: ragged={D1.is_ragged}")

# Step 2: SlidingWindow (ragged -> dense)
D2 = SlidingWindow(size=50, stride=25, label_strategy="repeat").fit_transform(D1)
print(f"After SlidingWindow: shape={D2.shape}")
print(f"Total windows: {D2.n_samples}, y expanded: len={len(D2.y)}")

In [ ]:
# Step 3: FeatureExtractor with custom autocorr
fe = FeatureExtractor(
    functions=[("mean", np.mean), ("std", np.std), ("lag1", autocorr_lag1)],
    input_dim=Dim.TIME,
)
D3 = fe.fit_transform(D2)
print(f"After FeatureExtractor: shape={D3.shape}")

# Step 4: Flatten
D4 = Flatten().fit_transform(D3)
print(f"After Flatten: shape={D4.shape} (3 features x 4 channels = {3*4})")
print(f"y present: {D4.y is not None}")

---
## 10. Split Utilities

In [ ]:
from tensorts import temporal_split, stratified_temporal_split, kfold_temporal

# temporal_split
data_sp = np.random.randn(100, 50, 3)
ts_sp = TimeTensor(data=data_sp, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                   y=np.arange(100))
train_s, test_s = temporal_split(ts_sp, test_ratio=0.2)
print(f"temporal_split: train={train_s.n_samples}, test={test_s.n_samples}")
print(f"Train y: [{train_s.y[0]}..{train_s.y[-1]}]")
print(f"Test y:  [{test_s.y[0]}..{test_s.y[-1]}]")
print(f"Order preserved: {np.array_equal(train_s.y, np.arange(80))}")

In [ ]:
# stratified_temporal_split
data_st = np.random.randn(100, 50, 3)
y_st = np.array([0] * 50 + [1] * 50)
ts_st = TimeTensor(data=data_st, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL), y=y_st)
train_st, test_st = stratified_temporal_split(ts_st, test_ratio=0.2)

print(f"stratified_temporal_split: train={train_st.n_samples}, test={test_st.n_samples}")
print(f"Test class 0: {np.sum(test_st.y == 0)}, class 1: {np.sum(test_st.y == 1)}")
print(f"Balanced: {np.sum(test_st.y == 0) == np.sum(test_st.y == 1)}")

In [ ]:
# kfold_temporal
data_kf = np.random.randn(50, 20, 2)
ts_kf = TimeTensor(data=data_kf, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
                   y=np.arange(50))
folds = kfold_temporal(ts_kf, n_folds=5)

print(f"Number of folds: {len(folds)}")
fig, ax = plt.subplots(figsize=(10, 4))
for i, (train_f, test_f) in enumerate(folds):
    no_leakage = train_f.y.max() < test_f.y.min()
    print(f"  Fold {i}: train={train_f.n_samples} [{train_f.y[0]}..{train_f.y[-1]}], "
          f"test={test_f.n_samples} [{test_f.y[0]}..{test_f.y[-1]}], no_leakage={no_leakage}")
    # Plot fold boundaries
    ax.barh(i, train_f.n_samples, left=0, color='steelblue', alpha=0.7, label='Train' if i == 0 else None)
    ax.barh(i, test_f.n_samples, left=train_f.n_samples, color='coral', alpha=0.7, label='Test' if i == 0 else None)

ax.set_xlabel("Samples")
ax.set_ylabel("Fold")
ax.set_title("kfold_temporal — Fold Boundaries (no future leakage)")
ax.legend()
ax.set_yticks(range(5))
plt.tight_layout()
plt.show()

---
All sections explored. The notebook demonstrates every major `tensorts` API with step-by-step execution and intermediate visualization.